In [2]:

import glob
import numpy as np
import nibabel as nib

In [6]:
fdir = "/media/storage/UT_subjects/proj-6542d7ceb094062da61faac2/sub-20231130AH/dt-neuro-rois.tag-tractEndpointDensity.qsiprep_freesurfer.id-65df6763f1e6dcc04088d276/rois"


In [4]:

def get_center(img):
    d = img.get_fdata()
    xs, ys, zs = np.where(d)
    return {"x": xs.mean(), "y": ys.mean(), "z": zs.mean()}


In [10]:

## define the major orientation of the tracts
tract_orientations = {
    "MDLFang":"y", 
    "MDLFspl":"y", 
    "Uncinate":"z", 
    "Aslant":"y"
}

for tname, orientation in tract_orientations.items():
    print(f"====={tname}=====")
    fnames = {}
    imgs = {}

    for fn in glob.glob(f"{fdir}/*"):
        if ((f"left{tname}" in fn) and ("RAS" in fn)):
            # print(fn)
            imgs["lras"] = nib.load(fn)
            fnames["lras"] = fn
        elif ((f"left{tname}" in fn) and ("LPI" in fn)):
            # print(fn)
            imgs["llpi"] = nib.load(fn)
            fnames["llpi"] = fn
        elif ((f"right{tname}" in fn) and ("RAS" in fn)):
            # print(fn)
            imgs["rras"] = nib.load(fn)
            fnames["rras"] = fn
        elif ((f"right{tname}" in fn) and ("LPI" in fn)):
            # print(fn)
            imgs["rlpi"] = nib.load(fn)
            fnames["rlpi"] = fn
    if len(imgs) < 4: 
        raise Exception(f"missing data for {tname} (only loaded {list(imgs.keys())}). Aborted.")

    centers = {tlab: get_center(img) for tlab, img in imgs.items()}
    for lab, cs in centers.items():
        print(lab, cs)

    ## sanity check: left x > right x
    print("=== Sanity check: left x > right x for all left/right pairs ===")
    for llab in ["lras", "llpi"]:
        for rlab in ["rras", "rlpi"]:
            if centers[llab]["x"] < centers[rlab]["x"]:
                raise Exception(f"{llab} {rlab} failed: data in different coordinates. Aborted.")
    print("passed")

    ## check each major orientation
    print(f"=== Checking ras {orientation} > lpi {orientation} ===")
    if orientation == "y": 
        ## y: S - I (vertical) --> ras < lpi
        
        for raslab, lpilab in [["lras", "llpi"], ["rras", "rlpi"]]:
            print(raslab, lpilab)
            if centers[raslab][orientation] < centers[lpilab][orientation]:
                print("True")
            else: 
                print("swapping ras/lpi ")
                rasfn = fnames[raslab]
                lpifn = fnames[lpilab]
                tempfn = f"{fdir}/temp"
                print("reveresing fnames")
                os.rename(rasfn, tempfn)
                os.rename(lpifn, rasfn)
                os.rename(tempfn, lpifn)
        
    if orientation == "z": 
        ## z: P - A (horizontal) --> ras > lpi
        
        for raslab, lpilab in [["lras", "llpi"], ["rras", "rlpi"]]:
            print(raslab, lpilab)
            if centers[raslab][orientation] > centers[lpilab][orientation]:
                print("True")
            else: 
                print("swapping ras/lpi ")
                rasfn = fnames[raslab]
                lpifn = fnames[lpilab]
                tempfn = f"{fdir}/temp"
                print("reveresing fnames")
                os.rename(rasfn, tempfn)
                os.rename(lpifn, rasfn)
                os.rename(tempfn, lpifn)


=====MDLFang=====
llpi {'x': 176.50372881355932, 'y': 117.77966101694915, 'z': 103.44542372881357}
lras {'x': 174.98693467336685, 'y': 166.04020100502512, 'z': 150.81105527638192}
rlpi {'x': 74.73853484216795, 'y': 128.73496128648006, 'z': 106.38653960690887}
rras {'x': 80.72861586314153, 'y': 165.5077760497667, 'z': 149.90513219284603}
=== Sanity check: left x > right x for all left/right pairs ===
passed
=== Checking ras y > lpi y ===
lras llpi
swapping ras/lpi 
rras rlpi
swapping ras/lpi 
=====MDLFspl=====
rras {'x': 100.96015362457993, 'y': 103.26404224675949, 'z': 89.36917906865098}
rlpi {'x': 82.51791530944625, 'y': 168.36400651465797, 'z': 151.21824104234528}
llpi {'x': 173.94162679425838, 'y': 167.41148325358853, 'z': 151.43253588516745}
lras {'x': 149.4575807334428, 'y': 100.55500821018063, 'z': 89.17077175697865}
=== Sanity check: left x > right x for all left/right pairs ===
passed
=== Checking ras y > lpi y ===
lras llpi
True
rras rlpi
True
=====Uncinate=====
llpi {'x': 165